# ORCA - QLoRA on Kaggle (CUDA OOM-hard mode)

Ultra-safe for ~16GB when normal settings still OOM.

- max_seq_length=**768**
- LoRA **r=8**, modules **q_proj+v_proj only**
- fp16 + gradient_checkpointing + paged_adamw_8bit
- 2 epochs, no eval

Setup: GPU + Internet On + Add Input (train.jsonl) -> Run All


In [ ]:
import gc
import torch

def cuda_clean():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('Enable GPU then Restart session')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
cuda_clean()


In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf


## 1. Find train.jsonl under /kaggle/input


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists(), 'Add Input dataset first'

train_path = None
found = list(INPUT_ROOT.rglob('*.jsonl'))
print('Found:', [str(p) for p in found])
for p in found:
    if p.name.lower() == 'train.jsonl':
        train_path = p
if train_path is None:
    for p in found:
        if 'train' in p.name.lower():
            train_path = p
            break
if train_path is None and found:
    train_path = found[0]
assert train_path is not None, 'train.jsonl missing'
print('train:', train_path)
print('N:', sum(1 for line in open(train_path, encoding='utf-8') if line.strip()))


## 2. Load train only (optional TRAIN_LIMIT)


In [ ]:
from datasets import load_dataset

TRAIN_LIMIT = None  # e.g. 100 if still OOM

raw = load_dataset('json', data_files={'train': str(train_path)})
if TRAIN_LIMIT is not None:
    raw['train'] = raw['train'].select(range(min(TRAIN_LIMIT, len(raw['train']))))
print(raw)


## 3. 4-bit model + LoRA r=8 (q_proj, v_proj only)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

cuda_clean()
BASE = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
cuda_clean()
print('Allocated GB:', round(torch.cuda.memory_allocated() / 1e9, 2))
print('Reserved GB:', round(torch.cuda.memory_reserved() / 1e9, 2))


## 4. Train seq=768, 2 epochs


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )

args = SFTConfig(
    output_dir='/kaggle/working/orca-400-out',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy='epoch',
    eval_strategy='no',
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    max_seq_length=768,
    packing=False,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    max_grad_norm=0.3,
    group_by_length=True,
)

cuda_clean()
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=raw['train'],
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
try:
    trainer.train()
    print('TRAIN DONE')
except torch.cuda.OutOfMemoryError as e:
    print('STILL OOM:', e)
    print('Set TRAIN_LIMIT=80, max_seq_length=512, Restart session')
    raise


## 5. Save zip


In [ ]:
from pathlib import Path
import shutil
OUT = Path('/kaggle/working/orca-analyst-lora')
if OUT.exists():
    shutil.rmtree(OUT)
model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))
!cd /kaggle/working && zip -r orca-analyst-lora.zip orca-analyst-lora
!ls -lh /kaggle/working/orca-analyst-lora.zip


## Still OOM?
1. **Restart session** (mandatory after OOM)
2. TRAIN_LIMIT = 100
3. max_seq_length = 512
4. Or use a 24GB GPU (RunPod) for one short job
